# Un Sistema per Sincronizzare i File tra Due Cartelle </p> Caso d'Uso Aziendale: SyncEase

## Introduzione all'Azienda
SyncEase è una società tecnologica che sviluppa soluzioni software per migliorare la gestione e la condivisione dei dati aziendali.
Una delle loro sfide principali è quella di garantire che i file critici siano sempre sincronizzati tra dispositivi e server, per evitare perdite di dati o versioni obsolete.

## Problema
Molte aziende si affidano a sistemi manuali o software non ottimizzati per mantenere sincronizzati i file tra due cartelle, ad esempio tra un server locale e uno di backup.
Questo approccio è soggetto a errori, come la perdita di file aggiornati o la duplicazione di dati. 
Inoltre, il processo può essere lento, specialmente quando il volume di dati è elevato.

## Obiettivo del Progetto
L'obiettivo è sviluppare un sistema software che utilizzi il multiprocessing e il multithreading in Python per sincronizzare in modo efficiente i file tra due cartelle. Questo sistema dovrà:

1. Copiare i file nuovi o modificati dalla cartella sorgente a quella di destinazione.
2. Eliminare i file obsoleti nella cartella di destinazione se non esistono più nella sorgente.
3. Gestire grandi volumi di file in modo rapido ed efficiente, sfruttando al meglio le risorse hardware.

## Benefici Attesi
- **Riduzione degli errori**: Sincronizzazione affidabile e automatica senza intervento manuale.
- **Ottimizzazione delle prestazioni**: Utilizzo efficiente di CPU e I/O grazie al multiprocessing e al multithreading.
- **Miglioramento della sicurezza**: Backup sempre aggiornati e consistenti tra le due cartelle.
   
## Specifiche del Progetto
1. **Input**:
   - Due percorsi di cartelle: source_folder e destination_folder.
2. **Output**:
   - La cartella di destinazione è sincronizzata con quella sorgente.
3. **Funzionalità Chiave**:
   - Utilizzo di multithreading per accelerare la lettura e la scrittura dei file.
   - Utilizzo di multiprocessing per gestire in parallelo sottogruppi di file.
   - Verifica di:
     - File nuovi: Copia solo i file che non esistono nella cartella di destinazione.
     - File modificati: Aggiorna i file nella destinazione se quelli nella sorgente sono più recenti.
     - File obsoleti: Rimuove i file nella destinazione che non esistono più nella sorgente.
  
## Consegna
Scrivi un programma Python che implementi il sistema di sincronizzazione dei file.
Il tuo codice deve essere ben documentato, con commenti che spieghino chiaramente le scelte implementative e il flusso del programma.
Testa il suo funzionamento con almeno 3 casi reali. Per esempio: - Una cartella con pochi file. - Una cartella con molti file. - Una cartella che include sottocartelle.

---

## Importazione delle Librerie

In questa sezione vengono importate tutte le librerie necessarie:

In [19]:
import os
import shutil
import hashlib
import time
import tempfile
import random
import string
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from typing import List, Tuple

## Funzioni di Utility

Prima di copiare un file, ne viene calcolato l'**hash MD5** per verificare se il contenuto è effettivamente cambiato rispetto alla destinazione. Questo evita copie inutili quando i file sono già identici. La verifica viene eseguita nella funzione principale `synchronize`, così la lista dei file da copiare è già definitiva prima di avviare i worker.

Vengono definite anche le funzioni per ottenere i percorsi relativi di tutti i file in una cartella (incluse le sottocartelle), per copiare i file e per rimuovere quelli obsoleti.


In [20]:
def compute_md5(filepath: str, chunk_size: int = 8192) -> str:
    """Calcola l'hash MD5 di un file leggendolo in chunk per gestire file grandi."""
    hasher = hashlib.md5()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def file_needs_update(src: str, dst: str) -> bool:
    """
    Determina se il file sorgente deve sovrascrivere quello di destinazione.
    Se la destinazione non esiste, la copia è necessaria.
    Altrimenti, confronta gli hash MD5 per rilevare modifiche al contenuto.
    """
    if not os.path.exists(dst):
        return True
    return compute_md5(src) != compute_md5(dst)


def get_relative_paths(folder: str) -> set:
    """
    Restituisce il set di tutti i percorsi relativi dei file in una cartella,
    esplorandola ricorsivamente per includere le sottocartelle.
    """
    result = set()
    for root, _, files in os.walk(folder):
        for filename in files:
            abs_path = os.path.join(root, filename)
            rel_path = os.path.relpath(abs_path, folder)
            result.add(rel_path)
    return result


def split_into_batches(items: list, n_batches: int) -> List[list]:
    """
    Divide una lista di elementi in n_batches sottoliste di dimensione uniforme.
    Utilizzata per distribuire il lavoro tra i processi/worker.
    """
    items = list(items)
    if not items:
        return []
    batch_size = max(1, len(items) // n_batches)
    return [items[i:i + batch_size] for i in range(0, len(items), batch_size)]


def copy_file(src_path: str, dst_path: str) -> None:
    """
    Copia un singolo file dalla sorgente alla destinazione.
    Crea le directory intermedie se non esistono (necessario per le sottocartelle).
    """
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    shutil.copy2(src_path, dst_path)


def delete_file(dst_path: str) -> None:
    """
    Elimina un file obsoleto dalla cartella di destinazione.
    Rimuove anche le directory vuote rimaste dopo l'eliminazione.
    """
    os.remove(dst_path)
    parent = os.path.dirname(dst_path)
    try:
        os.removedirs(parent)
    except OSError:
        pass  # La directory non è vuota, nessun problema


## Principali funzioni di sincronizzazione

La funzione `synchronize` orchestra l'intero processo in due livelli:

| Tecnologia                              | Scopo                                              |
| --------------------------------------- |----------------------------------------------------|
| `ProcessPoolExecutor` (multiprocessing) | Distribuisce i batch di file su più core della CPU |
| `ThreadPoolExecutor` (multithreading)   | Copia/elimina file in parallelo per ogni batch     |

**Flusso di esecuzione:**

1. Scansione della cartella sorgente e destinazione salvando i percorsi relativi
2. Calcolo MD5 per ogni file sorgente identificando i file da copiare (nuovi o modificati) e da eliminare
3. Divisione del lavoro in batch
4. Distribuzione dei batch a più processi tramite `ProcessPoolExecutor`
5. Ogni processo gestisce il proprio batch con un pool di thread interni per copiare/eliminare più file contemporaneamente `sync_fille_batch`


In [21]:
def sync_file_batch(
        batch: List[Tuple[str, str]],
        source_folder: str,
        destination_folder: str
) -> None:
    """
    Sincronizza un batch di file usando un ThreadPoolExecutor interno.
    Ogni elemento del batch è una tupla (azione, percorso_relativo)
    """

    max_threads = min(16, (os.cpu_count() or 1) * 2)

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = []
        for action, rel_path in batch:
            if action == "copy":
                src = os.path.join(source_folder, rel_path)
                dst = os.path.join(destination_folder, rel_path)
                futures.append(executor.submit(copy_file, src, dst))
            elif action == "delete":
                dst = os.path.join(destination_folder, rel_path)
                futures.append(executor.submit(delete_file, dst))

        for future in as_completed(futures):
            future.result()

In [22]:
def synchronize(source_folder: str, destination_folder: str) -> dict:
    """
    Sincronizza i file dalla cartella sorgente alla cartella destinazione.
    Ritorna un dizionario con le statistiche della sincronizzazione
    """
    source_folder = str(Path(source_folder).resolve())
    destination_folder = str(Path(destination_folder).resolve())

    src_files = get_relative_paths(source_folder)
    dst_files = get_relative_paths(destination_folder)

    # Il controllo MD5 avviene qui: solo i file effettivamente diversi entrano in to_copy
    to_copy = [
        ("copy", f)
        for f in src_files
        if file_needs_update(
            os.path.join(source_folder, f), os.path.join(destination_folder, f)
        )
    ]
    to_delete = [("delete", f) for f in (dst_files - src_files)]
    all_tasks = to_copy + to_delete

    stats = {
        "copiati": len(to_copy),
        "eliminati": len(to_delete),
        "invariati": len(src_files) - len(to_copy),
    }

    if not all_tasks:
        print("Nessuna operazione necessaria: le cartelle sono già sincronizzate.")
        return stats

    n_processes = min(os.cpu_count() or 1, len(all_tasks))
    batches = split_into_batches(all_tasks, n_processes)
    worker_args = [(b, source_folder, destination_folder) for b in batches]

    with ProcessPoolExecutor(max_workers=n_processes) as pool:
        futures = [pool.submit(sync_file_batch, *args) for args in worker_args]
        for future in as_completed(futures):
            future.result()

    return stats

---

## 5. Ambiente di Test

Per verificare il funzionamento del sistema vengono definite alcune funzioni di supporto:

- **`create_test_files`**: crea una struttura di file e directory a partire da un dizionario 
- **`random_content`**: genera testo casuale di dimensione arbitraria
- **`assert_folders_equal`**: confronta ricorsivamente due cartelle per verificare se contengano gli stessi file con lo stesso contenuto

In [23]:
def create_test_files(base_dir: str, files: dict) -> None:
    """
    Crea file di test nella cartella specificata
    """
    for rel_path, content in files.items():
        abs_path = os.path.join(base_dir, rel_path)
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(content)


def random_content(size: int = 200) -> str:
    """
    Genera una stringa casuale di caratteri stampabili di lunghezza 'size'
    """
    chars = string.ascii_letters + string.digits
    return "".join(random.choices(chars, k=size))


def assert_folders_equal(src: str, dst: str) -> None:
    """
    Verifica che le due cartelle abbiano esattamente gli stessi file (per percorso relativo).
    """
    src_files = get_relative_paths(src)
    dst_files = get_relative_paths(dst)
    missing = src_files - dst_files
    extra = dst_files - src_files
    assert not missing, f"File mancanti nella destinazione: {missing}"
    assert not extra, f"File extra nella destinazione:    {extra}"
    print("Le cartelle sono perfettamente sincronizzate")


## Test 1 — Cartella con pochi file

**Scenario**: cartella sorgente con 3 file di tipi diversi. La destinazione contiene già un file obsoleto (`old_report.txt`) che non esiste nella sorgente e deve essere eliminato.

In [24]:
with tempfile.TemporaryDirectory() as tmp:
    src = os.path.join(tmp, "source")
    dst = os.path.join(tmp, "destination")
    os.makedirs(src)
    os.makedirs(dst)

    # Sorgente
    create_test_files(src, {
        "report.txt": "Report mensile 2024",
        "config.json": '{"version": "2.1", "env": "production", "debug": false}',
        "readme.md": "Sistema di sincronizzazione file aziendali",
    })

    # Destinazione
    create_test_files(dst, {
        "old_report.txt": "Contenuto da eliminare",
    })

    stats = synchronize(src, dst)

    print(f"File copiati  : {stats['copiati']}")
    print(f"File eliminati: {stats['eliminati']}")
    print(f"File invariati: {stats['invariati']}")

    assert_folders_equal(src, dst)

File copiati  : 3
File eliminati: 1
File invariati: 0
Le cartelle sono perfettamente sincronizzate


## Test 2 — Cartella con molti file

**Scenario**: cartella sorgente con **100 file** generati casualmente. La destinazione ha già 20 file (con contenuto diverso, da aggiornare) e 5 file obsoleti (da eliminare).

In [25]:
with tempfile.TemporaryDirectory() as tmp:
    src = os.path.join(tmp, "source")
    dst = os.path.join(tmp, "destination")
    os.makedirs(src)
    os.makedirs(dst)

    # Sorgente: 100 file con contenuto casuale
    src_files_dict = {f"file_{i:03d}.dat": random_content() for i in range(100)}
    create_test_files(src, src_files_dict)

    # Destinazione: 20 file già esistenti ma con contenuto vecchio (da aggiornare)
    dst_old = {f"file_{i:03d}.dat": "Contenuto obsoleto — versione precedente" for i in range(20)}
    # Destinazione: 5 file extra da eliminare
    dst_old.update({f"obsoleto_{i}.tmp": f"File temporaneo {i} da rimuovere" for i in range(5)})
    create_test_files(dst, dst_old)

    stats = synchronize(src, dst)
    
    print(f"File copiati  : {stats['copiati']}")
    print(f"File eliminati: {stats['eliminati']}")
    print(f"File invariati: {stats['invariati']}")
    
    assert_folders_equal(src, dst)


File copiati  : 100
File eliminati: 5
File invariati: 0
Le cartelle sono perfettamente sincronizzate


## Test 3 — Cartella con sottocartelle

**Scenario**: struttura con sottocartelle annidate

In [26]:
with tempfile.TemporaryDirectory() as tmp:
    src = os.path.join(tmp, "source")
    dst = os.path.join(tmp, "destination")
    os.makedirs(src)
    os.makedirs(dst)

    create_test_files(src, {
        "README.md": "# Progetto SyncEase\nVersione 2.0",
        "documents/report_q1.txt": "Report Q1 2024 — Ricavi: 320.000€ (AGGIORNATO)",
        "documents/report_q2.txt": "Report Q2 2024 — Ricavi: 410.000€",
        "documents/presentazione.pptx": "[dati binari simulati] Presentazione board",
        "images/logo.png": "PNG\x00\x01\x02 fake binary content",
        "images/banner_home.jpg": "JFIF fake binary content",
        "data/raw/vendite_2024.csv": "id,data,importo\n1,2024-01-05,1500\n2,2024-01-12,2300",
        "data/raw/clienti.csv": "id,nome,citta\n1,Mario Rossi,Milano\n2,Anna Verdi,Roma",
        "data/processed/riepilogo.json": '{"totale_vendite": 3800, "clienti_attivi": 2}',
        "scripts/sync_runner.py": "#!/usr/bin/env python3\n# Avvia la sincronizzazione\n",
        "scripts/report_generator.py": "# Genera i report mensili\n",
    })
    
    create_test_files(dst, {
        "documents/report_q1.txt": "Report Q1 2023 — Ricavi: 290.000€ (VECCHIO)",
        "scripts/sync_runner.py": "#!/usr/bin/env python3\n# Avvia la sincronizzazione\n",
        "archive/dati_2022.csv": "dati storici 2022",
        "archive/backup_old.zip": "backup obsoleto",
    })

    stats = synchronize(src, dst)
    
    print(f"File copiati  : {stats['copiati']}")
    print(f"File eliminati: {stats['eliminati']}")
    print(f"File invariati: {stats['invariati']}")

    assert_folders_equal(src, dst)

File copiati  : 10
File eliminati: 2
File invariati: 1
Le cartelle sono perfettamente sincronizzate
